In [0]:
%pip install deepspeed pytorch-lightning transformers datasets
dbutils.library.restartPython()

In [0]:
import torch, socket
print("CUDA available:", torch.cuda.is_available())
print("GPUs on this node:", torch.cuda.device_count())
print("Device names:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print("Hostname:", socket.gethostname())

In [0]:
ddp_code = r"""
import torch
import torch.distributed as dist

def main():
    dist.init_process_group(backend="nccl")
    print(f"✅ Rank {dist.get_rank()} / World {dist.get_world_size()} is alive on {torch.cuda.get_device_name(0)}")
    dist.barrier()
    if dist.get_rank() == 0:
        print("🎉 Multi-node DDP test passed.")

if __name__ == "__main__":
    main()
"""
with open("/dbfs/ddp_test.py", "w") as f:
    f.write(ddp_code)
print("✅ ddp_test.py written.")

In [0]:
%sh
MASTER_ADDR=$(hostname -I | awk '{print $1}')
MASTER_PORT=29500
NCCL_SOCKET_IFNAME=eth0
torchrun --nnodes=2 --nproc_per_node=4 --rdzv_backend=c10d --rdzv_endpoint=$MASTER_ADDR:$MASTER_PORT /dbfs/ddp_test.py

In [0]:
ds_code = r"""
import deepspeed
import torch
import torch.nn as nn

class DummyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Linear(128, 128)

    def forward(self, x):
        return self.net(x)

def main():
    model = DummyModel()
    engine, optimizer, _, _ = deepspeed.initialize(
        model=model,
        model_parameters=model.parameters(),
        config="/dbfs/ds_config.json"
    )
    data = torch.randn(8, 128).to(engine.device)
    loss = engine(data).mean()
    engine.backward(loss)
    engine.step()
    if engine.global_rank == 0:
        print("🎉 DeepSpeed test step completed.")

if __name__ == "__main__":
    main()
"""
with open("/dbfs/ds_test.py", "w") as f:
    f.write(ds_code)

ds_config = {
    "train_batch_size": 128,
    "fp16": {"enabled": True},
    "zero_optimization": {"stage": 2}
}
import json
with open("/dbfs/ds_config.json", "w") as f:
    json.dump(ds_config, f, indent=2)

print("✅ ds_test.py and ds_config.json written.")

In [0]:
%sh
MASTER_ADDR=$(hostname -I | awk '{print $1}')
MASTER_PORT=29500
NCCL_SOCKET_IFNAME=eth0
deepspeed --num_nodes=2 --num_gpus=4 /dbfs/ds_test.py --deepspeed --deepspeed_config /dbfs/ds_config.json

In [0]:
gpt2_code = r"""
import os, torch
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.strategies import DeepSpeedStrategy
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

torch.set_float32_matmul_precision("high")

class GPT2Finetuner(pl.LightningModule):
    def __init__(self, model_name="gpt2", lr=5e-5):
        super().__init__()
        self.save_hyperparameters()
        self.model = GPT2LMHeadModel.from_pretrained(model_name)
        self.tokenizer = GPT2Tokenizer.from_pretrained(model_name)
        self.tokenizer.pad_token = self.tokenizer.eos_token
        ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
        def encode(ex):
            return self.tokenizer(ex["text"], truncation=True, padding="max_length", max_length=128)
        self.ds = ds.map(encode, batched=True).with_format("torch")

    def training_step(self, batch, _):
        out = self.model(input_ids=batch["input_ids"], labels=batch["input_ids"])
        loss = out.loss
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=5e-5)

    def train_dataloader(self):
        return DataLoader(self.ds, batch_size=8, shuffle=True, num_workers=2)

def main():
    strategy = DeepSpeedStrategy(config="/dbfs/ds_config.json")
    trainer = Trainer(
        accelerator="gpu",
        devices=1,              # TorchDistributor launches 1 proc per GPU
        strategy=strategy,
        precision="bf16-mixed",
        max_epochs=1
    )
    model = GPT2Finetuner()
    trainer.fit(model)

if __name__ == "__main__":
    main()
"""
with open("/dbfs/train_gpt2.py", "w") as f:
    f.write(gpt2_code)
print("✅ train_gpt2.py written.")

In [0]:
from pyspark.ml.torch.distributor import TorchDistributor

def entrypoint():
    import train_gpt2
    train_gpt2.main()
    return 0

TorchDistributor(num_processes=4, local_mode=False, use_gpu=True).run(entrypoint)